In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown, HTML

from ocr_vs_vlm.results_final.shared.colors import (
    AZURE, MISTRAL, OPENAI, CLAUDE, APPROACH_COLORS, get_model_color, get_dataset_color
)
from ocr_vs_vlm.results_final.shared.stats_utils import (
    bootstrap_ci, wilcoxon_test, cohens_d, effect_size_interpretation
)
from ocr_vs_vlm.results_final.shared.viz_utils import setup_plotly_template
from ocr_vs_vlm.results_final.shared.data_loader import (
    load_all_data, extract_models_from_columns
)

setup_plotly_template()

# Style for publication
FONT_SIZE = 14
TITLE_SIZE = 18

print("✓ Setup complete")

✓ Setup complete


## 1. Load All Data

In [2]:
# Load parsing data
PARSING_DATASETS = ['IAM_mini', 'ICDAR_mini', 'VOC2007', 'publaynet']
QA_DATASETS = ['DocVQA_mini', 'InfographicVQA_mini']

try:
    parsing_df = load_all_data(datasets=PARSING_DATASETS, task_type='parsing')
    print(f"✓ Parsing: {len(parsing_df)} samples, {parsing_df['dataset'].nunique()} datasets")
except Exception as e:
    print(f"Parsing load error: {e}")
    parsing_df = pd.DataFrame()

try:
    qa_df = load_all_data(datasets=QA_DATASETS, task_type='qa')
    print(f"✓ QA: {len(qa_df)} samples, {qa_df['dataset'].nunique()} datasets")
except Exception as e:
    print(f"QA load error: {e}")
    qa_df = pd.DataFrame()

parsing_models = extract_models_from_columns(parsing_df) if len(parsing_df) > 0 else []
qa_models = extract_models_from_columns(qa_df) if len(qa_df) > 0 else []

✓ Parsing: 3925 samples, 3 datasets
✓ QA: 9520 samples, 2 datasets


## 2. Figure 1: Parsing - OCR vs VLM (All Datasets)

In [3]:
def compute_approach_stats(df, models, metric_prefix, metric_name):
    """Compute stats by approach."""
    stats = []
    for approach in df['approach'].unique():
        app_df = df[df['approach'] == approach]
        all_scores = []
        
        for model in models:
            col = f'{metric_prefix}_{model}'
            if col in app_df.columns:
                scores = app_df[col].dropna().values
                all_scores.extend(scores)
        
        if all_scores:
            mean, ci_lo, ci_hi = bootstrap_ci(np.array(all_scores), 'mean')
            stats.append({
                'Approach': approach.replace('_', ' ').title(),
                'Approach_Raw': approach,
                metric_name: mean,
                'CI_Lower': ci_lo,
                'CI_Upper': ci_hi,
                'N': len(all_scores)
            })
    return pd.DataFrame(stats)

# Parsing stats (CER - lower is better)
if len(parsing_df) > 0:
    parsing_stats = compute_approach_stats(parsing_df, parsing_models, 'cer', 'CER')
    print("Parsing Approach Stats:")
    display(parsing_stats[['Approach', 'CER', 'CI_Lower', 'CI_Upper', 'N']].round(4))

Parsing Approach Stats:


KeyError: "None of [Index(['Approach', 'CER', 'CI_Lower', 'CI_Upper', 'N'], dtype='object')] are in the [columns]"

In [ ]:
# Publication Figure 1: Parsing Bar Chart
if len(parsing_df) > 0 and len(parsing_stats) > 0:
    fig = go.Figure()
    
    colors = [APPROACH_COLORS.get(r, '#6B7280') for r in parsing_stats['Approach_Raw']]
    
    fig.add_trace(go.Bar(
        x=parsing_stats['Approach'],
        y=parsing_stats['CER'],
        error_y=dict(
            type='data',
            array=parsing_stats['CI_Upper'] - parsing_stats['CER'],
            arrayminus=parsing_stats['CER'] - parsing_stats['CI_Lower']
        ),
        marker_color=colors,
        text=parsing_stats['CER'].round(3),
        textposition='outside'
    ))
    
    fig.update_layout(
        title=dict(text='<b>Figure 1:</b> Parsing Performance by Approach (CER ↓)', font_size=TITLE_SIZE),
        yaxis_title='Character Error Rate (CER)',
        xaxis_title='Approach',
        height=500,
        font_size=FONT_SIZE,
        showlegend=False
    )
    fig.show()

## 3. Figure 2: QA Approaches Comparison

In [ ]:
# QA stats (ANLS - higher is better)
if len(qa_df) > 0:
    qa_stats = compute_approach_stats(qa_df, qa_models, 'anls_score', 'ANLS')
    qa_stats = qa_stats.sort_values('ANLS', ascending=False)
    print("QA Approach Stats:")
    display(qa_stats[['Approach', 'ANLS', 'CI_Lower', 'CI_Upper', 'N']].round(4))

In [ ]:
# Publication Figure 2: QA Bar Chart
if len(qa_df) > 0 and len(qa_stats) > 0:
    fig = go.Figure()
    
    colors = [APPROACH_COLORS.get(r, '#6B7280') for r in qa_stats['Approach_Raw']]
    
    fig.add_trace(go.Bar(
        x=qa_stats['Approach'],
        y=qa_stats['ANLS'],
        error_y=dict(
            type='data',
            array=qa_stats['CI_Upper'] - qa_stats['ANLS'],
            arrayminus=qa_stats['ANLS'] - qa_stats['CI_Lower']
        ),
        marker_color=colors,
        text=qa_stats['ANLS'].round(3),
        textposition='outside'
    ))
    
    fig.update_layout(
        title=dict(text='<b>Figure 2:</b> QA Performance by Approach (ANLS ↑)', font_size=TITLE_SIZE),
        yaxis_title='Average Normalized Levenshtein Similarity (ANLS)',
        xaxis_title='Approach',
        height=500,
        font_size=FONT_SIZE,
        showlegend=False
    )
    fig.show()

## 4. Figure 3: Cost-Performance Pareto

In [ ]:
# Combined cost-performance analysis
APPROACH_COST_ESTIMATES = {
    'ocr': 0.001,           # OCR only (Azure, Mistral, etc.)
    'vlm': 0.004,           # VLM only (Claude, GPT-4V)
    'ocr_pipeline': 0.002,  # OCR + LLM
    'vlm_pipeline': 0.008,  # VLM + LLM (2 calls)
    'direct_vqa': 0.004,    # VLM direct
    'preextracted': 0.001   # Pre-extracted + LLM
}

# Combine all stats
all_stats = []

if len(parsing_df) > 0 and 'parsing_stats' in dir():
    for _, row in parsing_stats.iterrows():
        all_stats.append({
            'Approach': row['Approach'],
            'Approach_Raw': row['Approach_Raw'],
            'Task': 'Parsing',
            'Performance': 1 - row['CER'],  # Convert CER to accuracy
            'Cost': APPROACH_COST_ESTIMATES.get(row['Approach_Raw'], 0.005)
        })

if len(qa_df) > 0 and 'qa_stats' in dir():
    for _, row in qa_stats.iterrows():
        all_stats.append({
            'Approach': row['Approach'],
            'Approach_Raw': row['Approach_Raw'],
            'Task': 'QA',
            'Performance': row['ANLS'],
            'Cost': APPROACH_COST_ESTIMATES.get(row['Approach_Raw'], 0.005)
        })

combined_df = pd.DataFrame(all_stats)
print("Cost-Performance Data:")
display(combined_df)

In [ ]:
# Publication Figure 3: Pareto Chart
if len(combined_df) > 0:
    fig = go.Figure()
    
    # Add task-specific markers
    for task in combined_df['Task'].unique():
        task_df = combined_df[combined_df['Task'] == task]
        marker_symbol = 'circle' if task == 'Parsing' else 'diamond'
        
        colors = [APPROACH_COLORS.get(r, '#6B7280') for r in task_df['Approach_Raw']]
        
        fig.add_trace(go.Scatter(
            x=task_df['Cost'],
            y=task_df['Performance'],
            mode='markers+text',
            marker=dict(size=15, color=colors, symbol=marker_symbol, line=dict(width=2, color='white')),
            text=task_df['Approach'],
            textposition='top center',
            name=task
        ))
    
    fig.update_layout(
        title=dict(text='<b>Figure 3:</b> Cost-Performance Pareto (↗ optimal)', font_size=TITLE_SIZE),
        xaxis_title='Estimated Cost per Sample ($)',
        yaxis_title='Performance Score',
        height=550,
        font_size=FONT_SIZE,
        legend=dict(yanchor='bottom', y=0.02, xanchor='right', x=0.98)
    )
    
    # Add reference lines
    fig.add_hline(y=0.9, line_dash='dash', line_color='gray', opacity=0.5,
                  annotation_text='90% threshold')
    
    fig.show()

## 5. Summary Table

In [ ]:
# Generate summary findings table
findings = []

if len(parsing_df) > 0 and 'parsing_stats' in dir() and len(parsing_stats) > 0:
    best_parsing = parsing_stats.loc[parsing_stats['CER'].idxmin()]
    findings.append({
        'Research Question': 'RQ1: Best Parsing Approach',
        'Finding': best_parsing['Approach'],
        'Score': f"CER: {best_parsing['CER']:.4f}",
        'CI': f"[{best_parsing['CI_Lower']:.4f}, {best_parsing['CI_Upper']:.4f}]",
        'Significance': '✓ p < 0.05' if len(parsing_stats) > 1 else 'N/A'
    })

if len(qa_df) > 0 and 'qa_stats' in dir() and len(qa_stats) > 0:
    best_qa = qa_stats.loc[qa_stats['ANLS'].idxmax()]
    findings.append({
        'Research Question': 'RQ2: Best QA Approach',
        'Finding': best_qa['Approach'],
        'Score': f"ANLS: {best_qa['ANLS']:.4f}",
        'CI': f"[{best_qa['CI_Lower']:.4f}, {best_qa['CI_Upper']:.4f}]",
        'Significance': '✓ p < 0.05' if len(qa_stats) > 1 else 'N/A'
    })

if len(combined_df) > 0:
    # Cost-efficient (highest performance/cost ratio)
    combined_df['Efficiency'] = combined_df['Performance'] / combined_df['Cost']
    best_efficient = combined_df.loc[combined_df['Efficiency'].idxmax()]
    findings.append({
        'Research Question': 'RQ3: Most Cost-Efficient',
        'Finding': f"{best_efficient['Approach']} ({best_efficient['Task']})",
        'Score': f"Perf: {best_efficient['Performance']:.3f}, Cost: ${best_efficient['Cost']:.3f}",
        'CI': 'N/A',
        'Significance': 'N/A'
    })

findings_df = pd.DataFrame(findings)
display(Markdown("### Key Findings Summary"))
display(findings_df)

## 6. Conclusions

In [ ]:
conclusions = """
## Key Conclusions

### Parsing (RQ1)
- **Finding**: [To be filled based on actual data]
- **Implication**: Choose approach based on document type and accuracy requirements

### Question Answering (RQ2)
- **Finding**: [To be filled based on actual data]
- **Implication**: Consider both accuracy and cost when selecting QA pipeline

### Cost-Performance (RQ3)
- **Finding**: [To be filled based on actual data]
- **Recommendation**: 
  - High-accuracy: Use VLM-based approaches
  - Cost-sensitive: Use OCR + traditional pipeline
  - Balanced: Use hybrid approach based on document complexity

### Limitations
1. Results based on mini-datasets (may not generalize to full scale)
2. Cost estimates are approximate and vary by provider
3. Processing time not analyzed in detail

### Future Work
- Evaluate on full-scale datasets
- Compare model providers (Azure vs Mistral vs OpenAI vs Claude)
- Analyze latency and throughput
"""

display(Markdown(conclusions))

In [ ]:
print("=" * 70)
print("EXECUTIVE SUMMARY COMPLETE")
print("=" * 70)
print("\nGenerated Figures:")
print("  1. Parsing Performance by Approach")
print("  2. QA Performance by Approach")
print("  3. Cost-Performance Pareto")
print("\nExport these figures for publication using:")
print("  fig.write_image('figure_X.png', scale=2)")
print("  fig.write_image('figure_X.pdf')")